In [ ]:
# Cell 1: Imports and config
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import ipywidgets as widgets
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from IPython.display import display

os.chdir("/Users/matthew/Desktop/Pdu_PyRosetta")

# Positions: (protein, PDB_resnum, wt_aa)
POSITIONS = {
    "PduA_K37": ("PduA", 37, "K"),
    "PduA_S40": ("PduA", 40, "S"),
    "PduJ_K36": ("PduJ", 36, "K"),
    "PduJ_S39": ("PduJ", 39, "S"),
}

# H-bond donor amino acids
HBOND_DONORS = set("STNQHKRYW")

# Charge at physiological pH
AA_CHARGE = {
    "K": +1, "R": +1, "H": +0.1,
    "D": -1, "E": -1,
    "A": 0, "C": 0, "F": 0, "G": 0,
    "I": 0, "L": 0, "M": 0, "N": 0,
    "P": 0, "Q": 0, "S": 0, "T": 0,
    "V": 0, "W": 0, "Y": 0,
}

# Variants already tested experimentally by the user
ALREADY_VALIDATED = {"PduA_S40": {"S40A", "S40H", "S40Q", "S40L"}}

print("✓ Imports and config loaded")


In [ ]:
# Cell 2: Load all data sources
print("Loading docking summaries...")
docking = {}
DOCKING_PATHS = {
    "PduA_K37": "PduA_docking_mutants/K37_site_saturation_summary.csv",
    "PduA_S40": "PduA_docking_mutants/S40_site_saturation_summary.csv",
    "PduJ_K36": "PduJ_docking_mutants/K36_site_saturation_summary.csv",
    "PduJ_S39": "PduJ_docking_mutants/S39_site_saturation_summary.csv",
}
for pos_key, csv_path in DOCKING_PATHS.items():
    df = pd.read_csv(csv_path)
    df["position_key"] = pos_key
    docking[pos_key] = df
    print(f"  {pos_key}: {len(df)} mutants, ddg_mean [{df['ddg_mean'].min():.2f}, {df['ddg_mean'].max():.2f}]")

print("\nLoading SSM Rosetta \u0394\u0394G...")
ssm_a = pd.read_csv("PduA_SSM_results/PduA_SSM_ddg_rigorous.csv")
ssm_a = ssm_a[ssm_a["baseline_type"] == "local_relaxed"]
ssm_j = pd.read_csv("PduJ_SSM_results/PduJ_SSM_ddg_rigorous.csv")
ssm_j = ssm_j[ssm_j["baseline_type"] == "local_relaxed"]
print(f"  PduA SSM: {len(ssm_a)}, PduJ SSM: {len(ssm_j)}")

print("\nLoading ThermoMPNN stability predictions...")
thermo_a = pd.read_csv("PduA_ThermoMPNN_results/PduA_ThermoMPNN_combined.csv")
thermo_j = pd.read_csv("PduJ_SSM_results/PduJ_ThermoMPNN_zeroshot.csv")
thermo_j = thermo_j.rename(columns={"ddG_thermo": "ddG_ft"})

print("\nLoading conservation scores...")
cons = pd.read_csv("PduA_conservation_analysis/PduA_conservation_scores.csv")

print("\nLoading coevolution APC scores...")
coev = pd.read_csv("PduA_coevolution/top_coevolving_pairs.csv")
k37_s40_pair = coev[
    ((coev["res_i"] == 37) & (coev["res_j"] == 40)) |
    ((coev["res_i"] == 40) & (coev["res_j"] == 37))
]
APC_K37_S40 = float(k37_s40_pair["APC_score"].iloc[0]) if len(k37_s40_pair) > 0 else 0.1894
APC_K36_S39 = APC_K37_S40  # PduJ proxy (no coevolution file; same pore sequence)
print(f"  K37-S40 APC = {APC_K37_S40:.4f}  (K36-S39 proxy = {APC_K36_S39:.4f})")

print("\nLoading BindingHead predictions...")
binding_preds_path = "PduA_ThermoMPNN_results/binding_head_all_predictions.csv"
if os.path.exists(binding_preds_path):
    binding_preds = pd.read_csv(binding_preds_path)
    print(f"  {len(binding_preds)} predictions loaded")
else:
    print("  \u26a0 binding_head_all_predictions.csv not found yet \u2014 run BMC_ThermoMPNN_binding.py first")
    binding_preds = pd.DataFrame(columns=["position_key","mut_aa","ddg_binding_predicted"])

print("\n\u2713 All data loaded.")


In [ ]:
# Cell 3: Build unified feature dataframe per position

def build_feature_df(pos_key: str) -> pd.DataFrame:
    protein, pdb_resnum, wt_aa = POSITIONS[pos_key]

    # 1. Docking scores
    doc = docking[pos_key].copy()
    doc["mut_aa"] = doc["mutant"].str[-1]
    doc["wt_aa"]  = wt_aa
    doc = doc.rename(columns={
        "ddg_mean":      "ddg_binding",
        "ddg_best":      "ddg_binding_best",
        "ddg_top10":     "ddg_binding_top10",
        "std_interface": "ddg_binding_std",
    })

    # 2. SSM structural \u0394\u0394G
    ssm = ssm_a if protein == "PduA" else ssm_j
    ssm_pos = ssm[ssm["residue_num"] == pdb_resnum][["mut_aa", "ddG_rigorous_REU"]].copy()
    ssm_pos = ssm_pos.rename(columns={"ddG_rigorous_REU": "ddg_structural"})

    # 3. ThermoMPNN stability
    thermo = thermo_a if protein == "PduA" else thermo_j
    thermo_pos = thermo[thermo["residue_num"] == pdb_resnum][["mut_aa", "ddG_ft"]].copy()

    # 4. BindingHead predictions
    bp = binding_preds[binding_preds["position_key"] == pos_key][["mut_aa", "ddg_binding_predicted"]].copy()

    # 5. Conservation: homolog frequency of mut_aa
    cons_pos = cons[cons["residue_num"] == pdb_resnum]
    def get_freq(mut_aa):
        col = f"freq_{mut_aa}"
        if not cons_pos.empty and col in cons_pos.columns:
            return float(cons_pos[col].iloc[0])
        return 0.0

    # Merge
    keep_cols = ["mutant", "mut_aa", "wt_aa", "ddg_binding", "ddg_binding_best",
                 "ddg_binding_top10", "ddg_binding_std", "mean_pore_dist"]
    keep_cols = [c for c in keep_cols if c in doc.columns]
    df = doc[keep_cols].copy()
    df = df.merge(ssm_pos,    on="mut_aa", how="left")
    df = df.merge(thermo_pos, on="mut_aa", how="left")
    df = df.merge(bp,         on="mut_aa", how="left")

    df["homolog_freq"]    = df["mut_aa"].apply(get_freq)
    df["charge_mut"]      = df["mut_aa"].map(AA_CHARGE).fillna(0)
    df["hbond_donor_mut"] = df["mut_aa"].apply(lambda a: 1 if a in HBOND_DONORS else 0)
    df["charge_wt"]       = AA_CHARGE.get(wt_aa, 0)
    df["hbond_donor_wt"]  = 1 if wt_aa in HBOND_DONORS else 0
    df["delta_charge"]    = df["charge_mut"] - df["charge_wt"]
    df["delta_hbond"]     = df["hbond_donor_mut"] - df["hbond_donor_wt"]
    df["position_key"]    = pos_key
    df["protein"]         = protein
    df["pdb_resnum"]      = pdb_resnum

    # Bootstrap 95% CI approx: mean \u00b1 1.96 * std / sqrt(100)
    df["bootstrap_ci_lo"] = df["ddg_binding"] - 1.96 * df["ddg_binding_std"] / 10
    df["bootstrap_ci_hi"] = df["ddg_binding"] + 1.96 * df["ddg_binding_std"] / 10

    df["already_validated"] = df["mutant"].isin(ALREADY_VALIDATED.get(pos_key, set()))
    return df.reset_index(drop=True)

feature_dfs = {pk: build_feature_df(pk) for pk in POSITIONS}
for pk, df in feature_dfs.items():
    print(f"{pk}: {len(df)} variants, ddg_binding range [{df['ddg_binding'].min():.2f}, {df['ddg_binding'].max():.2f}]")
print("\u2713 Feature dataframes built")


In [ ]:
# ============================================================
# Cell 4: Composite Score — USER CONTRIBUTION POINT
# ============================================================
# WHY THIS MATTERS:
# The relative weights encode your scientific prior about what predicts
# experimental success.
#   - High ddg_binding weight → picks strong binders that may be unstable
#   - High ddg_structural weight → picks stable variants that may not change binding
#   - High homolog_freq weight → conservative; prioritizes natural variants
#
# CONSTRAINTS:
#   - All weights must be >= 0 and sum to 1.0
#   - Z-scoring is done within each position (across the 19 variants)
#   - Lower composite score = better candidate
#   - homolog_freq is INVERTED before Z-scoring (higher freq = lower z = better)
# ============================================================

# Adjust these weights to reflect your scientific priorities:
WEIGHTS = {
    "ddg_binding":           0.40,   # direct binding evidence (docking \u0394\u0394G)
    "ddg_binding_predicted": 0.20,   # BindingHead model prediction
    "ddg_structural":        0.25,   # apo hexamer stability (SSM Rosetta \u0394\u0394G)
    "ddg_ft":                0.10,   # ThermoMPNN stability score
    "homolog_freq":          0.05,   # evolutionary support
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-6, "Weights must sum to 1.0"


def z_normalize_features(df: pd.DataFrame) -> pd.DataFrame:
    """Z-score each feature within the position's variant set.
    homolog_freq is inverted so lower z = better for all features.
    """
    df = df.copy()
    for col in ["ddg_binding", "ddg_binding_predicted", "ddg_structural", "ddg_ft"]:
        if col in df.columns:
            mu, sigma = df[col].mean(), df[col].std()
            df[f"z_{col}"] = (df[col] - mu) / (sigma + 1e-9)
    if "homolog_freq" in df.columns:
        mu, sigma = df["homolog_freq"].mean(), df["homolog_freq"].std()
        df["z_homolog_freq"] = -1.0 * (df["homolog_freq"] - mu) / (sigma + 1e-9)
    return df


def compute_composite_score(row: pd.Series, weights: dict) -> float:
    """Weighted composite score. Lower = better candidate.

    Handles missing features by renormalizing available weights.
    Caps Z-scores at \u00b13 to prevent extreme values from dominating.
    """
    score = 0.0
    available_weight = 0.0
    for feature, w in weights.items():
        z_col = f"z_{feature}"
        if z_col in row.index and not pd.isna(row[z_col]):
            z_val = float(np.clip(row[z_col], -3, 3))
            score += w * z_val
            available_weight += w
    if available_weight > 0:
        score = score / available_weight
    return score


# Verify
test_df = z_normalize_features(feature_dfs["PduA_K37"])
test_score = compute_composite_score(test_df.iloc[0], WEIGHTS)
print(f"Test composite score (PduA_K37 row 0): {test_score:.3f}")
print(f"Z-columns present: {[c for c in test_df.columns if c.startswith('z_')]}")
print("\u2713 compute_composite_score ready \u2014 adjust WEIGHTS above to tune rankings")


In [ ]:
# Cell 5: Single mutant rankings for all 4 positions

def rank_single_mutants(pos_key: str, weights: dict) -> pd.DataFrame:
    df = z_normalize_features(feature_dfs[pos_key])
    df["composite_score"] = df.apply(compute_composite_score, axis=1, weights=weights)
    df = df.sort_values("composite_score").reset_index(drop=True)
    df["rank"] = df.index + 1
    df["top5"]  = df["rank"] <= 5
    return df

all_single_rankings = {}
for pos_key in POSITIONS:
    ranked = rank_single_mutants(pos_key, WEIGHTS)
    all_single_rankings[pos_key] = ranked
    ranked.to_csv(f"mutant_candidates/{pos_key}_single_mutant_ranking.csv", index=False)

    display_cols = ["rank", "mutant", "ddg_binding", "ddg_structural", "ddG_ft",
                    "homolog_freq", "composite_score", "already_validated", "top5"]
    show_cols = [c for c in display_cols if c in ranked.columns]
    print(f"\n{'='*60}")
    print(f"  {pos_key} \u2014 Single Mutant Rankings")
    print(f"{'='*60}")
    print(ranked[show_cols].to_string(index=False))

print("\n\u2713 Single mutant rankings saved to mutant_candidates/")

# Quick check: validated PduA S40 variants should appear
validated = {"S40A", "S40H", "S40Q", "S40L"}
s40_df = all_single_rankings["PduA_S40"]
found = set(s40_df["mutant"]) & validated
print(f"\nPduA S40 validated variants found: {found}")
assert found == validated, f"Missing: {validated - found}"
print("\u2713 All validated variants present in rankings")


In [ ]:
# ============================================================
# Cell 6: Epistasis Lambda — USER CONTRIBUTION POINT
# ============================================================
# WHY THIS MATTERS:
# K37-S40 is the only confirmed pore-pore coevolving pair (both_pore=True,
# APC = 0.1894 from the coevolution analysis). Lambda controls how much this
# evolutionary signal up-weights double mutant predictions beyond the additive model.
#
#   lambda = 0:   purely additive (ddG_double = ddG_K + ddG_S)
#   lambda = 0.5: moderate APC upweighting (recommended starting point)
#   lambda = 1.0: strong APC effect
#   lambda = 2.0: APC dominates over additive signal
#
# SCIENTIFIC NOTE: Positive APC = positions co-vary in evolution = they likely
# interact. A pair where BOTH mutations favor binding gets amplified by APC
# weighting (ddG_additive is negative → more negative after scaling by 1+λ*APC).
# ============================================================

lambda_widget = widgets.FloatSlider(
    value=0.5, min=0.0, max=2.0, step=0.1,
    description="APC λ:",
    continuous_update=False,
    style={"description_width": "60px"},
    layout=widgets.Layout(width="400px"),
)
display(lambda_widget)
print(f"K37-S40 APC = {APC_K37_S40:.4f}  |  K36-S39 APC (proxy) = {APC_K36_S39:.4f}")
print("Adjust λ above. Double mutant predictions in Cell 7 will use this value.")


In [ ]:
# Cell 7: Double mutant predictions

def predict_double_mutants(protein: str, lam: float = None) -> pd.DataFrame:
    """All 19×19 double mutant combinations for K+S positions.

    For PduA: K37×S40. For PduJ: K36×S39.

    ddG_epistatic = (ddG_K + ddG_S) * (1 + lam * APC_score)

    Filter criteria:
        - Both ddg_binding < 0      (both individually improve binding)
        - Both ddg_structural < +5  (not too destabilizing; NaN passes)
        - Both homolog_freq > 0.02  (seen in ≥2% of homologs)

    Returns full 361-row DataFrame sorted ascending by ddg_epistatic_score.
    """
    if lam is None:
        lam = lambda_widget.value

    if protein == "PduA":
        k_key, s_key = "PduA_K37", "PduA_S40"
        apc = APC_K37_S40
    else:
        k_key, s_key = "PduJ_K36", "PduJ_S39"
        apc = APC_K36_S39

    k_df = all_single_rankings[k_key][
        ["mutant", "mut_aa", "ddg_binding", "ddg_structural", "homolog_freq"]
    ].copy()
    s_df = all_single_rankings[s_key][
        ["mutant", "mut_aa", "ddg_binding", "ddg_structural", "homolog_freq"]
    ].copy()

    rows = []
    for _, kr in k_df.iterrows():
        for _, sr in s_df.iterrows():
            ddg_additive  = kr["ddg_binding"] + sr["ddg_binding"]
            ddg_epistatic = ddg_additive * (1.0 + lam * apc)

            passes = (
                kr["ddg_binding"]  < 0 and
                sr["ddg_binding"]  < 0 and
                (pd.isna(kr["ddg_structural"]) or kr["ddg_structural"] < 5.0) and
                (pd.isna(sr["ddg_structural"]) or sr["ddg_structural"] < 5.0) and
                kr["homolog_freq"] > 0.02 and
                sr["homolog_freq"] > 0.02
            )
            rows.append({
                "protein":             protein,
                "K_mut":               kr["mutant"],
                "S_mut":               sr["mutant"],
                "double_mutant":       f"{kr['mutant']}+{sr['mutant']}",
                "ddg_K":               kr["ddg_binding"],
                "ddg_S":               sr["ddg_binding"],
                "ddg_additive":        ddg_additive,
                "APC_score":           apc,
                "lambda":              lam,
                "ddg_epistatic_score": ddg_epistatic,
                "ddg_K_structural":    kr["ddg_structural"],
                "ddg_S_structural":    sr["ddg_structural"],
                "homolog_freq_K":      kr["homolog_freq"],
                "homolog_freq_S":      sr["homolog_freq"],
                "passes_filters":      passes,
            })

    df = pd.DataFrame(rows).sort_values("ddg_epistatic_score").reset_index(drop=True)
    df["rank"] = df.index + 1
    return df


double_pduA = predict_double_mutants("PduA")
double_pduJ = predict_double_mutants("PduJ")

for protein, df in [("PduA", double_pduA), ("PduJ", double_pduJ)]:
    df.to_csv(f"mutant_candidates/{protein}_double_mutant_predictions.csv", index=False)
    top20 = df[df["passes_filters"]].head(20)
    top20.to_csv(f"mutant_candidates/{protein}_double_mutant_top20.csv", index=False)

    print(f"\n{'='*60}")
    print(f"  {protein} Double Mutants  (λ={lambda_widget.value})")
    print(f"{'='*60}")
    print(f"Total: {len(df)} combinations  |  Passing filters: {df['passes_filters'].sum()}")
    print(f"\nTop 5 (passing filters):")
    show = top20[["rank","double_mutant","ddg_additive","ddg_epistatic_score","passes_filters"]].head(5)
    print(show.to_string(index=False))

print("\n✓ Double mutant predictions saved to mutant_candidates/")


In [ ]:
# ============================================================
# Cell 8: Propionaldehyde Selectivity — USER CONTRIBUTION POINT
# ============================================================
# BACKGROUND:
# Propionaldehyde (PA) lacks the -OH groups of 1,2-PD. Two physical factors
# affect PA transit through the pore:
#
# 1. PORE SIZE (Δmin_pore_dist): Wider pore → PA passes more freely
#    (dominant effect for K37 mutations which alter charge but not H-bond capacity)
#
# 2. H-BOND CAPACITY (delta_hbond): When S40 loses H-bond donor capacity
#    (e.g. S→A, delta_hbond=-1), 1,2-PD binding is selectively weakened
#    because PA cannot H-bond anyway. This makes the pore LESS selective
#    for 1,2-PD → PA passage is relatively enhanced.
#
# FORMULA: PA_index = alpha * Δpore - beta * delta_hbond
#   - Positive PA_index → mutation ENHANCES PA passage (less selective for 1,2-PD)
#   - Negative PA_index → mutation RESTRICTS PA passage (more selective for 1,2-PD)
#
# Recommended starting values: alpha=1.0, beta=0.5
# (pore size is the primary effect; H-bond loss is secondary)
#
# ADJUST THESE:
ALPHA = 1.0   # weight for pore size change
BETA  = 0.5   # weight for H-bond capacity change

# WT pore distances (mean min_pore_distance for the WT protein)
# Loaded from WT docking files if available; else estimated from mutant means
wt_pore_dists = {}
for pos_key in POSITIONS:
    protein, pdb_resnum, wt_aa = POSITIONS[pos_key]
    wt_file = (f"PduA_docking_mutants/WT/docking_scores.csv" if protein == "PduA"
               else f"PduJ_docking_mutants/WT/docking_scores.csv")
    if os.path.exists(wt_file):
        wt_df = pd.read_csv(wt_file)
        wt_pore_dists[pos_key] = float(wt_df["min_pore_distance"].mean())
    else:
        # Fallback: use mean of mutant pore distances as WT approximation
        wt_pore_dists[pos_key] = float(feature_dfs[pos_key]["mean_pore_dist"].mean())
print("WT pore distances (mean min_pore_dist, Å):", wt_pore_dists)


def propionaldehyde_selectivity(row: pd.Series, wt_pore_dist: float) -> float:
    """Propionaldehyde selectivity index for one mutant.

    Args:
        row: pandas Series with mean_pore_dist and delta_hbond columns
        wt_pore_dist: mean min_pore_distance for the WT protein at this position

    Returns:
        float: PA_selectivity_index
               > 0 → mutation enhances PA passage (less selective for 1,2-PD)
               < 0 → mutation restricts PA passage (more selective for 1,2-PD)
    """
    delta_pore  = row["mean_pore_dist"] - wt_pore_dist    # positive = wider pore
    delta_hbond = row["delta_hbond"]                       # -1 = H-bond lost, 0 = same, +1 = H-bond gained
    # Wider pore → more PA passage (positive)
    # Lost H-bond → 1,2-PD weakened more than PA → PA relatively enhanced (positive)
    return float(ALPHA * delta_pore - BETA * delta_hbond)


# Test
test_row = feature_dfs["PduA_S40"].iloc[0].copy()
test_val = propionaldehyde_selectivity(test_row, wt_pore_dists["PduA_S40"])
print(f"Test PA selectivity (PduA_S40 row 0 = {test_row['mutant']}): {test_val:.3f}")
print("✓ propionaldehyde_selectivity ready — adjust ALPHA and BETA above")


In [ ]:
# Cell 9: Compute propionaldehyde and propionate selectivity for all positions

def propionate_selectivity(row: pd.Series) -> float:
    """Propionate is a negatively charged anion (-1 at pH 7).

    K37 (+1 charge) electrostatically EXCLUDES propionate.
    PR_selectivity index = change in propionate passage tendency.

    Positive PR_index → mutation increases propionate passage
    Negative PR_index → mutation decreases propionate passage (more exclusion)
    """
    delta_charge = row["delta_charge"]   # charge(mut) - charge(wt)
    # K37→A: delta_charge = 0 - (+1) = -1 → charge lost → propionate passes more
    # K37→R: delta_charge = (+1) - (+1) = 0 → no change
    # K37→D: delta_charge = (-1) - (+1) = -2 → strongest propionate passage
    # S40→K: delta_charge = (+1) - 0 = +1 → new positive charge → propionate excluded more
    return float(-delta_charge)  # negate: lost positive charge = more passage = positive index


def selectivity_label(index_val: float, threshold: float = 0.3) -> str:
    if index_val > threshold:
        return "enhanced passage"
    elif index_val < -threshold:
        return "enhanced exclusion"
    else:
        return "neutral"


all_selectivity_dfs = {}
for pos_key in POSITIONS:
    protein, pdb_resnum, wt_aa = POSITIONS[pos_key]
    df = feature_dfs[pos_key].copy()
    wt_pd_val = wt_pore_dists[pos_key]

    df["wt_pore_dist"]   = wt_pd_val
    df["delta_pore_dist"] = df["mean_pore_dist"] - wt_pd_val

    df["PA_selectivity_index"] = df.apply(
        propionaldehyde_selectivity, axis=1, wt_pore_dist=wt_pd_val
    )
    df["PR_selectivity_index"] = df.apply(propionate_selectivity, axis=1)
    df["PA_effect"]            = df["PA_selectivity_index"].apply(selectivity_label)
    df["PR_effect"]            = df["PR_selectivity_index"].apply(selectivity_label)

    sel_cols = ["mutant", "position_key", "protein", "pdb_resnum",
                "ddg_binding", "PA_selectivity_index", "PA_effect",
                "PR_selectivity_index", "PR_effect",
                "delta_pore_dist", "delta_charge", "delta_hbond"]
    sel_df = df[[c for c in sel_cols if c in df.columns]].copy()

    # Save PA selectivity
    sel_df.sort_values("PA_selectivity_index").to_csv(
        f"mutant_candidates/{protein}_{pos_key.split('_')[1]}_propionaldehyde_selectivity.csv",
        index=False
    )
    # Save PR selectivity
    sel_df.sort_values("PR_selectivity_index").to_csv(
        f"mutant_candidates/{protein}_{pos_key.split('_')[1]}_propionate_selectivity.csv",
        index=False
    )
    all_selectivity_dfs[pos_key] = sel_df

    print(f"\n{pos_key} — top PA passage (least selective for 1,2-PD):")
    top_pa = sel_df.nlargest(3, "PA_selectivity_index")
    print(top_pa[["mutant","PA_selectivity_index","PA_effect","PR_selectivity_index","PR_effect"]].to_string(index=False))

print("\n✓ Selectivity predictions saved (8 CSV files)")


In [ ]:
# Cell 10: Final summary report

def get_top_n(pos_key: str, n: int = 3) -> list:
    return all_single_rankings[pos_key].head(n)["mutant"].tolist()

summary_rows = []
for protein in ["PduA", "PduJ"]:
    k_key = f"{protein}_K37" if protein == "PduA" else f"{protein}_K36"
    s_key = f"{protein}_S40" if protein == "PduA" else f"{protein}_S39"
    k_label = "K37" if protein == "PduA" else "K36"
    s_label = "S40" if protein == "PduA" else "S39"

    s_top3 = get_top_n(s_key, 3)
    k_top3 = get_top_n(k_key, 3)

    double_df  = double_pduA if protein == "PduA" else double_pduJ
    double_top3 = double_df[double_df["passes_filters"]].head(3)["double_mutant"].tolist()

    # Best PA exclusion (most negative PA_selectivity_index across both positions)
    pa_data = []
    for pk in [k_key, s_key]:
        sel = all_selectivity_dfs[pk]
        for _, r in sel.iterrows():
            pa_data.append((r["mutant"], r["PA_selectivity_index"]))
    pa_best = min(pa_data, key=lambda x: x[1])

    # Best PR exclusion
    pr_data = []
    for pk in [k_key, s_key]:
        sel = all_selectivity_dfs[pk]
        for _, r in sel.iterrows():
            pr_data.append((r["mutant"], r["PR_selectivity_index"]))
    pr_best = min(pr_data, key=lambda x: x[1])

    summary_rows.extend([
        {"Category": f"{s_label} single (binding ↑)", "Protein": protein,
         "Top candidates": ", ".join(s_top3)},
        {"Category": f"{k_label} single (binding ↑)", "Protein": protein,
         "Top candidates": ", ".join(k_top3)},
        {"Category": f"{k_label}+{s_label} double (binding ↑)", "Protein": protein,
         "Top candidates": " | ".join(double_top3) if double_top3 else "none pass filters"},
        {"Category": "PA exclusion (more 1,2-PD selective)", "Protein": protein,
         "Top candidates": f"{pa_best[0]} (PA_idx={pa_best[1]:.2f})"},
        {"Category": "PR exclusion (blocks propionate)", "Protein": protein,
         "Top candidates": f"{pr_best[0]} (PR_idx={pr_best[1]:.2f})"},
    ])

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("mutant_candidates/final_summary.csv", index=False)

print("=" * 72)
print("  FINAL MUTANT CANDIDATE SUMMARY")
print("=" * 72)
print(summary_df.to_string(index=False))

print(f"\n✓ Summary saved to mutant_candidates/final_summary.csv")
print("\nAll output files:")
for f in sorted(Path("mutant_candidates").iterdir()):
    print(f"  {f.name}")
